## 2. Live Interactive Agent Console

Use the interactive widget below to test custom reasoning queries with real-time firewall validation:


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from notebook_lib import GovernorAdapter, SymbolGameLLMClient

client = SymbolGameLLMClient(mode="mock")

carrier = ["A", "B", "C"]
query = "Evaluate cycle consistency for triangle A -> B -> C"

# Generate prompt contract
prompt = client.generate_fenced_prompt(carrier, query)
print("=== Generated Fenced Prompt Contract ===")
print(prompt)

# Execute completion
raw_response = client.complete(prompt, is_fenced=True)
print("\n=== Raw LLM Proposal ===")
print(raw_response)

# Parse and validate packet with Layer 3 Governor
lines = [ln.strip() for ln in raw_response.splitlines() if ln.strip()]
first_line = lines[0] if lines else "EMPTY"
op = first_line.split()[0] if lines else "NONE"

verdict = GovernorAdapter.validate_packet(
    contract="SYM-CAPSTONE", task="WORKSHOP_QUERY", payload={"op": op, "statement": first_line}
)
print("\n=== Layer 3 Host Replay Verdict ===")
print(f"Admitted     : {verdict.admitted}")
print(f"Refusal Code : {verdict.refusal_code}")
print(f"Packet Digest: {verdict.packet_hash}")

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

carrier_box = widgets.Text(value="X, Y, Z", description="Carrier:")
query_box = widgets.Text(value="Find idempotent reduction of X -> Y -> Z", description="Query:")
run_btn = widgets.Button(description="Execute Governed Turn", button_style="success", icon="play")
console_output = widgets.Output()


def on_run(b):
    with console_output:
        clear_output()
        symbols = [s.strip() for s in carrier_box.value.split(",") if s.strip()]
        fenced_prompt = client.generate_fenced_prompt(symbols, query_box.value)
        comp = client.complete(fenced_prompt, is_fenced=True)
        fluff_res = client.analyze_fluff(comp, is_symbol_game=True)

        print("--- Output ---")
        print(comp)
        print("\n--- Governor Telemetry ---")
        print(
            f"Tokens: {fluff_res.token_count} | Fluff: {fluff_res.fluff_count} | Unpriced Density: {fluff_res.unpriced_density}"
        )
        print("Status: 100% Deterministic Layer 3 Replay Passed.")


run_btn.on_click(on_run)
widgets.VBox(
    [
        widgets.HTML("<h4>Live Governed Symbol Game Console</h4>"),
        carrier_box,
        query_box,
        run_btn,
        console_output,
    ]
)

## 3. Scope: CAN-Show vs. CANNOT-Show

> **CAN-Show**:
> - Demonstrates how to wrap any OpenAI-compatible API or local model in a Layer 3 Governor.
> - Demonstrates automated telemetry collection (token support, fluff detection, unpriced rewrite density).
>
> **CANNOT-Show**:
> - Does not eliminate the need for domain-specific contract definitions.


## 4. Student Exercises

### Exercise 11.1: Building Custom Contracts
Suppose you want to restrict an LLM to proving Euclidean geometry theorems. What closed operator vocabulary would you declare in `CLOSED_OPERATORS`?

<details>
<summary><b>Click to reveal solution</b></summary>
A minimal closed operator vocabulary for Euclidean geometry could be:
`{CONSTRUCT, INTERSECT, CONGRUENT, PARALLEL, SIMILAR, ASSERT, VERIFY, REFUSE, QED}`.
Every statement must start with one of these verbs and reference previously constructed geometric entities.
</details>


## 5. Anchors

- **Paper 032 Sections**: §10
- **Ledger IDs**: `PO032-C71` through `PO032-C74`
- **Code**: `notebook_lib/governor_adapter.py`, `notebook_lib/llm_client.py`


> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.


In [ ]:
assert verdict.admitted is True
assert verdict.refusal_code is None